# MARTA — Demo Fase 1: clasificación de cobertura de suelo

Trabajo Final de Maestría (Maestría en Inteligencia Artificial y Análisis de Datos, FP-UNA) — tutor **Rodrigo Parra**.

Este notebook demuestra **Fase 1 completa** de MARTA: dataset Sentinel-2 + MapBiomas, comparación
Random Forest vs. CNN, y el criterio de selección de modelo. Cada celda de hiperparámetros trae su
propia explicación — pensado para poder cambiar un valor en vivo y explicar qué hace.

Acompaña a `docs/RECORRIDO_PASO_A_PASO.md` (versión completa y narrativa) y
`docs/METODOLOGIA_PIPELINE.md` (referencia técnica) del repo.

## Setup (una sola vez)

Este notebook corre sobre el dataset ya armado (1.193 patches de 33×33 píxeles), no vuelve a bajar
nada de Google Earth Engine — así el demo es rápido y no depende de que el profesor tenga acceso a
un proyecto de GEE.

**Antes de la demo, una sola vez:**
1. Subí `notebooks/marta_colab_bundle.zip` (generado en el repo) a tu Google Drive, en
   `MyDrive/marta_demo/marta_colab_bundle.zip`.
2. Corré la celda de abajo — monta el Drive y descomprime el dataset + los módulos compartidos
   (`dataset_loader.py`, `classifier_report.py`, exactamente los mismos que usa el pipeline real,
   no una copia reescrita) a `/content/marta/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
from pathlib import Path

BUNDLE_ZIP = "/content/drive/MyDrive/marta_demo/marta_colab_bundle.zip"
CONTENT_ROOT = Path("/content")
REPO_ROOT = CONTENT_ROOT / "marta"

if not REPO_ROOT.exists():
    with zipfile.ZipFile(BUNDLE_ZIP) as z:
        z.extractall(CONTENT_ROOT)

n_patches = sum(1 for _ in (REPO_ROOT / "data/study_area/dataset").rglob("*.npy"))
print(f"Patches encontrados: {n_patches}")


In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "scripts"))

import dataset_loader as dl
import classifier_report

print("Clases:", dl.CLASS_NAMES)


---
## Paso 0 — Por qué existe este proyecto

En el Chaco paraguayo hay proyectos que venden "créditos de carbono": le prometen a una empresa
que pagándoles se evita la deforestación de un terreno, lo cual evita la liberación de CO2 a la
atmósfera. Para vender esos créditos, el proyecto tiene que demostrarle a un auditor cuánto bosque
tiene y cuánto carbono almacena.

El problema real **no es la ausencia de automatización** — ya existen herramientas de
clasificación satelital que corren a escala (Global Forest Watch, MapBiomas, usado oficialmente
por MADES). El problema es que ninguna le da al auditor **evidencia trazable por predicción
individual**: recibe un número final que tiene que creer, no evidencia que pueda revisar caso por
caso. Un estudio publicado en *Science* (2025) encontró que, en promedio, los proyectos REDD+ a
nivel mundial emiten 10,7 veces más créditos de los que en realidad se justifican.

**Pregunta del TFM**: ¿se puede construir, con imágenes satelitales gratuitas, un sistema que
clasifique cobertura de suelo con evidencia interpretable de cada predicción, y traduzca el cambio
detectado en toneladas de CO2 — de forma auditable? Este notebook cubre la primera mitad
(clasificar); la estimación de carbono es la Fase 2, todavía no implementada.

## Paso 1 — Qué imágenes usamos, y por qué

- **Imágenes: Sentinel-2 (Copernicus/ESA), nivel L2A** — 10m de resolución, 13 bandas espectrales
  (no solo rojo/verde/azul: incluye infrarrojo cercano y red-edge, donde la vegetación sana se
  distingue mejor de la estresada o el suelo desnudo). L2A (reflectancia de superficie, corregida
  atmosféricamente) y no L1C (tope de atmósfera) porque la comparación temporal 2019 vs. 2023
  necesita reflectancia comparable entre fechas — con L1C, condiciones atmosféricas distintas en
  cada fecha meterían diferencias que no son cambio real de cobertura.
- **Etiquetas: MapBiomas Chaco, Colección 5** — clasificación ya hecha, 1985-2023, específica para
  el Gran Chaco. Contra: 30m de resolución (viene de Landsat), 3 veces más grueso que el píxel de
  Sentinel-2 (retomado en el Paso 3).

In [ ]:
from collections import Counter

for split in ["train", "val", "test"]:
    rows = dl.load_manifest(split)
    counts = Counter(r["class"] for r in rows)
    print(f"{split:5s} ({len(rows):4d} patches):", {k: counts.get(k, 0) for k in dl.CLASS_NAMES})


## Paso 2 — Las 3 áreas de estudio

1. **AOI del caso de estudio**: Corazón Verde del Chaco (VCS 2611), ~20.515 ha — el terreno real
   que se auditaría en la Fase 4. No tiene ni un solo píxel de "cultivo" en ningún año.
2. **Huella de muestreo**: los 3 departamentos del Chaco paraguayo (Alto Paraguay, Boquerón,
   Presidente Hayes), 24 millones de hectáreas — de acá salen los 1.193 patches de entrenamiento,
   porque el AOI solo no tiene ejemplos de cultivo. Muestreo determinista (`seed=42`), no aleatorio
   entre corridas.
3. **Sub-área held-out: Filadelfia, Boquerón** — excluida a propósito de todo entrenamiento, para
   chequear generalización contra ESA WorldCover (una fuente que nunca se usó para entrenar, a
   diferencia de MapBiomas). *Este chequeo ya se corrió (`phase1-classifier` tarea 5): ahí RF le
   gana a la CNN por 0,136 F1 macro — al revés de lo que muestra el test split del propio AOI, más
   abajo. Retomado en el Paso 5.*

### Cómo se ve un patch, por clase

Color verdadero (bandas B4/B3/B2 = rojo/verde/azul), reflectancia de superficie escalada a
[0, 1] solo para visualización — el modelo entrena con los valores crudos, sin esta normalización.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

RGB_IDX = [3, 2, 1]  # B4, B3, B2 en S2_BANDS (ver scripts/s2_utils.py)
N_EXAMPLES = 4

fig, axes = plt.subplots(len(dl.CLASS_NAMES), N_EXAMPLES, figsize=(2.2 * N_EXAMPLES, 2.2 * len(dl.CLASS_NAMES)))
for row, class_name in enumerate(dl.CLASS_NAMES):
    examples = [r for r in dl.load_manifest("train") if r["class"] == class_name][:N_EXAMPLES]
    for col, r in enumerate(examples):
        patch = np.load(REPO_ROOT / r["path"])
        rgb = np.clip(patch[:, :, RGB_IDX] / 3000, 0, 1)
        ax = axes[row, col]
        ax.imshow(rgb)
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(class_name, fontsize=12)
fig.suptitle("Patches de ejemplo por clase (color verdadero)")
fig.tight_layout()
plt.show()


## Paso 3 — Cómo se arma un patch, y la fuga espacial que casi invalida el resultado

MapBiomas (30m) y Sentinel-2 (10m) tienen ratio exacto 3:1 — cada píxel de MapBiomas es un bloque
de 3×3 píxeles de Sentinel-2, la **unidad de alineación de la etiqueta**. El patch final es
**33×33** (no 3×3): un input de 9 píxeles apenas permite una convolución significativa, y le
quitaría sentido a la comparación CNN-vs-RF de más abajo — la CNN necesita contexto espacial real
para tener alguna chance de superar a un clasificador por-píxel.

**La fuga de datos (la parte más importante de esta sección)**: cada patch mide 330m de lado — dos
patches a menos de esa distancia real pueden solaparse en el terreno. Repartir puntos al azar entre
train/val/test puede mandar dos patches del mismo pedazo de campo a lados opuestos del split
(fuga), inflando artificialmente la métrica. Con una versión con fuga, la conclusión había sido
"RF gana claro" (0,826 vs 0,770 macro-F1); corregida, cambió a "empate técnico" — la celda de abajo
verifica en vivo que la corrección se sostiene.

In [ ]:
from scipy.spatial import cKDTree

def to_meters_xy(lon, lat, lat0):
    """Proyección equirectangular simple, suficiente para distancias cortas dentro del Chaco."""
    R = 6371000
    x = np.radians(lon) * R * np.cos(np.radians(lat0))
    y = np.radians(lat) * R
    return x, y

all_rows = dl.load_manifest()
lat0 = np.mean([float(r["lat"]) for r in all_rows])
coords = np.array([to_meters_xy(float(r["lon"]), float(r["lat"]), lat0) for r in all_rows])
splits = np.array([r["split"] for r in all_rows])

tree = cKDTree(coords)
PATCH_SIZE_M = 330
min_cross_split_dist = np.inf
for i, (pt, split) in enumerate(zip(coords, splits)):
    idx = tree.query_ball_point(pt, r=PATCH_SIZE_M * 6)
    for j in idx:
        if splits[j] != split and j != i:
            d = np.linalg.norm(coords[i] - coords[j])
            min_cross_split_dist = min(min_cross_split_dist, d)

print(f"Par más cercano entre splits distintos: {min_cross_split_dist:.0f} m")
print(f"Umbral de fuga (tamaño del patch): {PATCH_SIZE_M} m")
print("Sin fuga" if min_cross_split_dist >= PATCH_SIZE_M else "FUGA DETECTADA")


## Paso 4 — Los dos modelos, y cómo funcionan por dentro

**Random Forest**: cadena de árboles de decisión, cada uno preguntando sobre valores de banda de
un solo píxel ("¿B11 > umbral?"), promediando el voto de cientos de árboles entrenados sobre
muestras y subconjuntos de bandas distintos (bootstrap). No tiene ningún mecanismo que sepa que un
píxel está "al lado" de otro — mira exclusivamente valores espectrales.

**CNN**: desliza filtros pequeños (3×3) sobre el patch completo, construyendo patrones cada vez más
abstractos capa a capa — a diferencia de RF, sí puede aprovechar cómo se relacionan los píxeles
vecinos entre sí. Esa es la capacidad que el experimento de abajo pone a prueba.

### Random Forest — hiperparámetros

Los dos que se pueden tocar en vivo están en la celda de entrenamiento, como constantes al
principio (igual que en `scripts/09_train_random_forest.py`):

- **`N_ESTIMATORS` (400)** — cantidad de árboles. Cada árbol individual sobreajusta su propia
  muestra bootstrap; promediar cientos de árboles que se equivocan de formas distintas entre sí
  cancela buena parte de ese error. Rango típico 300-500 — subirlo más allá no suele mejorar
  nada (retornos decrecientes), bajarlo mucho (ej. 20) hace que el promedio sea menos estable
  y más sensible a la semilla.
- **`max_depth=None`** — profundidad máxima de cada árbol, sin límite. En un árbol *solo* esto
  sería peligroso (memoriza el ruido de sus datos); en un Random Forest no lo es, porque la
  reducción de sobreajuste viene de promediar árboles distintos entre sí, no de podar cada uno.
  Limitarlo (ej. a 10) entrena más rápido y usa menos memoria, a costa de algo de precisión —
  no hace falta a esta escala de datos.
- **`random_state` (la semilla)** — controla qué muestra bootstrap y qué subconjunto de bandas ve
  cada árbol. Cambiarla no debería cambiar mucho el resultado si el modelo es estable — es
  justamente lo que la comparación de 5 semillas del Paso 5 pone a prueba.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

N_ESTIMATORS = 400   # cambiá esto para la demo en vivo
MAX_DEPTH = None     # cambiá esto para la demo en vivo
SEED = 42

X_train, y_train = dl.load_pixel_features("train")
print("Distribución de clases en train:", Counter(dl.CLASS_NAMES[c] for c in y_train))

rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
print("Random Forest entrenado.")


In [ ]:
X_test, y_test = dl.load_pixel_features("test")
rf_pred = rf.predict(X_test)
rf_results = classifier_report.evaluate_predictions(y_test, rf_pred)
classifier_report.print_evaluation(rf_results, "Random Forest")

print("\nImportancia de bandas (B1..B12, NDVI):")
for name, imp in zip(["B1","B2","B3","B4","B5","B6","B7","B8","B8A","B9","B11","B12","NDVI"], rf.feature_importances_.round(3)):
    print(f"  {name}: {imp}")


### CNN — arquitectura y por qué es chica y propia, no un backbone pre-entrenado

Un backbone grande pre-entrenado (ej. ResNet en ImageNet) "sabe" texturas de fotos de gatos y
autos, no de vegetación, y espera 3 canales RGB, no 13 de reflectancia — habría que forzar la
primera capa y de todos modos la mayoría de sus pesos no vendrían de datos de este proyecto. Con
una CNN chica y propia, cada peso se entrenó con Sentinel-2 del Chaco: la pregunta "¿la
complejidad de la CNN se justifica con estos datos?" tiene una respuesta honesta.

In [ ]:
import torch
import torch.nn as nn

class SmallCNN(nn.Module):
    """3 bloques convolucionales (32->64->128 filtros) + pooling global + dropout + lineal.
    Idéntica a scripts/10_train_cnn.py — ver la celda de hiperparámetros de abajo."""

    def __init__(self, in_channels=13, n_classes=3, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, n_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)


### CNN — hiperparámetros, uno por uno

- **Kernel 3×3, `padding=1`** — 3×3 es el kernel más chico que ya mira más allá de un solo píxel;
  el padding mantiene el tamaño espacial después de cada convolución, para que sea el pooling
  (no una convolución sin padding) el único responsable de reducir resolución.
- **Filtros 32→64→128** — cuántos patrones distintos aprende cada bloque. Duplicar en cada bloque
  es la convención estándar (cada bloque reduce resolución espacial a la mitad vía pooling
  mientras duplica la profundidad de canales, manteniendo el cómputo balanceado). Más filtros =
  más capacidad para patrones complejos, pero más parámetros para sobreajustar con solo ~840
  ejemplos de entrenamiento — 32/64/128 es deliberadamente modesto para ese volumen.
- **`BatchNorm2d`** — normaliza las activaciones de cada canal dentro de un batch, estabiliza el
  entrenamiento y permite usar una tasa de aprendizaje más alta sin que diverja.
- **`MaxPool2d(2)` (x2)** — reduce la resolución espacial a la mitad cada vez, quedándose con la
  activación más fuerte de cada bloque de 2×2. De 33×33 termina en ~8×8 antes del pooling global.
- **`AdaptiveAvgPool2d(1)` (global average pooling)** — colapsa el mapa final a un solo vector de
  128 valores, promediando espacialmente cada canal. Evita una capa `Linear` gigante después de
  aplanar todo el mapa (muchos menos parámetros) y hace que el resultado no dependa de en qué
  posición exacta del patch apareció cada patrón.
- **`dropout` (0.4)** — apaga al azar el 40% de las neuronas justo antes de la capa final, solo
  durante entrenamiento (se desactiva automáticamente en `model.eval()`). Fuerza a que el modelo
  no dependa demasiado de un patrón puntual. Rango típico 0,3-0,5 para datasets chicos — subirlo
  mucho (ej. 0.7) puede hacer que el modelo no retenga suficiente señal (underfitting); bajarlo
  a 0 elimina esta regularización por completo.
- **`batch_size` (32)** — cuántos patches se procesan juntos antes de actualizar los pesos. Más
  grande = gradiente más estable pero menos actualizaciones por época y más memoria; más chico =
  gradiente más ruidoso, a veces ayuda a generalizar mejor con pocos datos. 32 da ~26 batches por
  época con las ~840 patches de train.
- **`learning_rate` (0.001)** — tamaño del paso en cada actualización de pesos (optimizador Adam).
  Muy alto → la pérdida diverge u oscila sin converger; muy bajo → el entrenamiento es lentísimo o
  se estanca en un mínimo pobre dentro del presupuesto de épocas. 0.001 es el valor por defecto
  recomendado en el paper original de Adam (Kingma & Ba, 2014).
- **`max_epochs` (100) + `early_stop_patience` (10)** — `max_epochs` es solo un techo de
  seguridad; la regla real es la parada temprana: si la pérdida de validación no mejora durante
  10 épocas seguidas, se frena y se recupera el mejor checkpoint guardado. Deja que el propio dato
  decida cuánto entrenar, en vez de adivinar un número fijo, y evita sobreajustar memorizando el
  training set más allá del punto donde deja de ayudar en validación.
- **`seed_everything`** — fija numpy, `random` y torch juntos. Importante: una revisión de código
  encontró que originalmente la semilla solo controlaba PyTorch, no el RNG de numpy que maneja el
  aumentado de datos (rotación/espejado) — así que una misma semilla no reproducía la misma
  corrida hasta que se corrigió esto.

In [ ]:
import random
from torch.utils.data import DataLoader

BATCH_SIZE = 32          # cambiá esto para la demo en vivo
LEARNING_RATE = 0.001    # cambiá esto para la demo en vivo
MAX_EPOCHS = 100
EARLY_STOP_PATIENCE = 10
DROPOUT = 0.4            # cambiá esto para la demo en vivo

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def run_epoch(model, loader, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()
    with torch.set_grad_enabled(train):
        for patches, labels in loader:
            patches, labels = patches.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            logits = model(patches)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            n += len(labels)
    return total_loss / n, correct / n

seed_everything(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

train_loader = DataLoader(dl.PatchDataset("train", augment=True), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dl.PatchDataset("val", augment=False), batch_size=BATCH_SIZE)

cnn = SmallCNN(dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(cnn.parameters(), lr=LEARNING_RATE)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss, patience_left, best_state = float("inf"), EARLY_STOP_PATIENCE, None
for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = run_epoch(cnn, train_loader, optimizer, device, train=True)
    val_loss, val_acc = run_epoch(cnn, val_loader, optimizer, device, train=False)
    for key, value in [("train_loss", train_loss), ("val_loss", val_loss), ("train_acc", train_acc), ("val_acc", val_acc)]:
        history[key].append(value)
    if epoch % 5 == 0:
        print(f"  epoch {epoch}: train_loss={train_loss:.3f} val_loss={val_loss:.3f} train_acc={train_acc:.3f} val_acc={val_acc:.3f}")

    if val_loss < best_val_loss:
        best_val_loss, patience_left = val_loss, EARLY_STOP_PATIENCE
        best_state = {k: v.clone() for k, v in cnn.state_dict().items()}
    else:
        patience_left -= 1
        if patience_left == 0:
            print(f"  parada temprana en la época {epoch} (mejor val_loss={best_val_loss:.3f})")
            break

cnn.load_state_dict(best_state)
print("CNN entrenada.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Pérdida"); axes[0].legend(); axes[0].set_xlabel("época")
axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("época")
fig.tight_layout()
plt.show()


In [ ]:
test_loader = DataLoader(dl.PatchDataset("test", augment=False), batch_size=BATCH_SIZE)
cnn.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for patches, labels in test_loader:
        logits = cnn(patches.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_labels.append(labels.numpy())
cnn_pred = np.concatenate(all_preds)
cnn_true = np.concatenate(all_labels)
cnn_results = classifier_report.evaluate_predictions(cnn_true, cnn_pred)
classifier_report.print_evaluation(cnn_results, "CNN")


## Paso 5 — El resultado real, y qué modelo usa MARTA (en revisión)

Con las 5 semillas ya corridas (fuera de este notebook, `scripts/11_compare_classifiers.py`,
persistido en `data/study_area/comparison_results.json`): **RF 0,735 F1 macro (± 0,006), CNN 0,759
(± 0,021)** — diferencia (+0,024) por encima del umbral pre-registrado (0,015, un desvío
combinado). **La CNN gana, esta vez sin ambigüedad**, en el test split del propio AOI. La corrida
individual de arriba, con una sola semilla, debería dar un número parecido a estos promedios.

In [ ]:
import json

with open(REPO_ROOT / "data/study_area/comparison_results.json") as f:
    comp = json.load(f)

fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([comp["rf_scores"], comp["cnn_scores"]])
ax.set_xticks([1, 2], ["Random Forest", "CNN"])
ax.set_ylabel("F1 macro (test)")
ax.set_title("5 semillas por modelo — split train/val/test fijo")
ax.axhline(comp["rf_mean"], color="gray", linestyle="--", linewidth=0.8)
plt.show()

print(f"RF:  media={comp['rf_mean']:.3f}  desvío={comp['rf_std']:.3f}")
print(f"CNN: media={comp['cnn_mean']:.3f}  desvío={comp['cnn_std']:.3f}")


**Por qué la CNN gana en el AOI, sin ser necesariamente la historia completa**:
bosque/pasto/cultivo en el Chaco se distinguen sobre todo por firma espectral (SWIR y red-edge,
arriba en la importancia de RF), pero el contexto espacial (33×33) sí puede aportar algo — surcos
de cultivo visibles a simple vista en la grilla del Paso 3 son un ejemplo concreto de señal
espacial real en los datos. Con el dataset corregido (sin fuga entre 2019/2023), la CNN aprovecha
esa señal mejor que RF en el propio test split del AOI.

**Pero Filadelfia (Paso 2) cuenta la historia contraria — y con margen todavía más grande**: ahí
RF le gana a la CNN por 0,136 F1 macro, con la CNN degradándose específicamente en bosque (IoU
0,803 → 0,451). Filadelfia es la única región genuinamente ajena a la huella de entrenamiento, así
que esta discrepancia no es un detalle menor: la CNN puede estar aprendiendo algo específico de la
geografía de entrenamiento que no generaliza, mientras que el umbral espectral simple de RF sí lo
hace.

**Cuál modelo usa MARTA en la práctica: todavía sin decidir** (`phase1-classifier`, tarea 4.4,
2026-09-11). La tarea 4.3 había designado a la CNN como clasificador operativo asumiendo que no
costaba nada en precisión frente a RF — ese supuesto ahora está en duda por el resultado de
Filadelfia. En vez de resolver esto con dos puntos de datos que se contradicen, la decisión queda
en revisión hasta conseguir más evidencia (ej. otra sub-área held-out independiente, o diagnosticar
por qué la CNN falla puntualmente en bosque fuera de la huella de entrenamiento).

## Qué sigue (fuera de este notebook)

- **Reconciliar RF vs. CNN como clasificador operativo** (`phase1-classifier` tarea 4.4) — el
  punto abierto más importante ahora mismo, ver Paso 5.
- **Grad-CAM**: evidencia visual por predicción sobre la CNN — todavía no implementado, change de
  OpenSpec propio. Su valor no depende de que la CNN gane en precisión, pero si la tarea 4.4
  termina prefiriendo RF, Grad-CAM pasaría a ser una herramienta de diagnóstico, no el clasificador
  de producción.
- **Fase 2**: estimación de carbono (altura de dosel pre-entrenada + calibración GEDI L4A +
  conversión IPCC a CO2e).